# Pipeline 5 — Safehouse Capacity / Incident Forecast

**Created:** 2026-04-06T16:55:25.900572Z

This notebook is one complete ML pipeline for INTEX (IS455). It is written so a TA can run it top-to-bottom.

## Required sections included
1. Problem Framing
2. Data Acquisition, Preparation & Exploration
3. Modeling & Feature Selection
4. Evaluation & Interpretation
5. Causal and Relationship Analysis
6. Deployment Notes


## 1) Problem Framing

**Business question:** Which safehouses are likely to face capacity strain or incident spikes next month?

**Who cares:** Leadership operations and safety oversight.

**Why it matters:** Prevents overload and safety failures by reallocating resources and planning staffing proactively.

**Prediction target:** Regression: next_month_active_residents and/or next_month_incident_count at safehouse-month level.

We will build:
- A **predictive model** optimized for out-of-sample performance (operational decision support).
- A **causal/explanatory model** optimized for interpretability to understand relationships (not causal proof unless assumptions hold).


## 2) Data Acquisition, Preparation & Exploration

Expected dataset location:
- Place CSVs under `data/raw/*.csv` (not committed with secrets).

This notebook is designed to be reproducible:
- All feature engineering is code-driven (no manual spreadsheet steps).
- All joins are documented in code.


In [ ]:
# Standard imports
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor

try:
    import statsmodels.api as sm
except Exception as e:
    sm = None
    print("statsmodels not available (optional for causal model). You can `pip install statsmodels` if needed.")


REPO_ROOT = Path("..").resolve() if Path("data").exists() is False else Path(".").resolve()
DATA_DIR = (REPO_ROOT / "data" / "raw").resolve()

print("Repo root:", REPO_ROOT)
print("Data dir:", DATA_DIR)


In [ ]:
def require_csv(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Place the provided INTEX CSVs under data/raw/ (e.g., data/raw/{name}.csv)."
        )
    return pd.read_csv(path)


# Load only what you need for this pipeline (add more tables as needed)
supporters = None
donations = None
social_media_posts = None
residents = None
education_records = None
health_wellbeing_records = None
incident_reports = None
safehouse_monthly_metrics = None

available = [p.stem for p in DATA_DIR.glob("*.csv")] if DATA_DIR.exists() else []
print("Available CSVs:", sorted(available)[:10], "..." if len(available) > 10 else "")


In [ ]:
def eval_classification(y_true, y_pred, y_proba=None):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    out = {"accuracy": acc, "precision": pr, "recall": rc, "f1": f1}
    if y_proba is not None:
        try:
            out["roc_auc"] = roc_auc_score(y_true, y_proba)
        except Exception:
            pass
    return out


def eval_regression(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred, squared=False)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def export_predictions_json(
    prediction_type: str,
    entity_type: str,
    entity_ids: pd.Series,
    scores: pd.Series,
    labels: pd.Series | None = None,
    payload_json: pd.Series | None = None,
    out_dir: Path | None = None,
):
    out_dir = out_dir or (REPO_ROOT / "output" / "ml-predictions")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{prediction_type}.json"

    rows = []
    for i in range(len(entity_ids)):
        rows.append(
            {
                "predictionType": prediction_type,
                "entityType": entity_type,
                "entityId": int(entity_ids.iloc[i]),
                "score": float(scores.iloc[i]),
                "label": None if labels is None else (None if pd.isna(labels.iloc[i]) else str(labels.iloc[i])),
                "payloadJson": None
                if payload_json is None
                else (None if pd.isna(payload_json.iloc[i]) else str(payload_json.iloc[i])),
            }
        )

    out_path.write_text(pd.Series(rows).to_json(orient="values", indent=2), encoding="utf-8")
    print("Wrote predictions:", out_path)


## 3) Modeling & Feature Selection

Modeling approach notes:
Use safehouse_monthly_metrics with lag features (t-1, t-2) and trend features. Predictive: RandomForestRegressor/GradientBoostingRegressor. Explanatory: linear regression on incident_count vs load + education/health aggregates.

We will implement at least:
- **Predictive**: Tree/ensemble (captures nonlinearities, interactions)
- **Explanatory**: Linear/Logistic regression with interpretable coefficients


In [ ]:
# TODO: Load the required tables for this pipeline.
# Example:
# donations = require_csv("donations")
# supporters = require_csv("supporters")


In [ ]:
# TODO: Define an "as-of" date for labeling to avoid leakage.
# Use only information that would have been known at that time.
AS_OF_DATE = None  # e.g., pd.Timestamp("2026-03-01")


In [ ]:
# TODO: Build a modeling dataframe at the correct grain (supporter / resident / safehouse / post).
# - Define the unit of analysis
# - Build features using only past data relative to AS_OF_DATE
# - Create label/target
df = None


In [ ]:
# TODO: Basic EDA checks (distributions, missingness, leakage checks)


In [ ]:
# TODO: Split data and build predictive model pipeline
# - Choose numeric/categorical features
# - Use ColumnTransformer for one-hot encoding
# - Fit and evaluate


In [ ]:
# TODO: Fit explanatory model
# - Prefer logistic/linear regression
# - Discuss coefficients / feature importances


## 4) Evaluation & Interpretation

We evaluate using metrics appropriate to the problem type.
- Classification: accuracy, precision, recall, F1, ROC-AUC (when possible)
- Regression: MAE, RMSE, R²

We interpret results in business terms (what to do next) and highlight cost of errors.


In [ ]:
# TODO: Evaluate and interpret results in business terms.


## 5) Causal and Relationship Analysis

We do **not** claim causality from predictive performance.

For explanatory modeling, we discuss:
- Which features appear most important
- Whether relationships make theoretical sense
- Where confounding / selection bias / leakage could exist


In [ ]:
# TODO: Relationship analysis and limitations.


## 6) Deployment Notes

Deploy as a monthly forecast feeding the Admin Dashboard: safehouses flagged for intervention. Add an endpoint `GET /api/ml/safehouse-forecast` returning risk/forecast bands.

### Import into the deployed app (Admin-only)

After exporting predictions JSON, import into the API:
- Endpoint: `POST /api/ml/import?replace=true`
- Body: the JSON array written by `export_predictions_json(...)`

Then view results in the web app:
- Staff portal → **ML Insights** (`/app/ml`)


In [ ]:
# TODO: Export predictions for app integration (example).
# Replace entity_ids/scores with your real outputs.
#
# export_predictions_json(
#     prediction_type="donor_lapse_90d",
#     entity_type="Supporter",
#     entity_ids=df["supporter_id"],
#     scores=df["risk_score"],
#     labels=df.get("risk_band"),
#     payload_json=df.get("explanations_json"),
# )
